In [7]:
import os
import numpy as np
import matplotlib.pyplot as plt

from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.applications.efficientnet import EfficientNetB0, preprocess_input

from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

In [8]:
def load_dataset(data_dir, split="train"):
    X, y = [], []
    classes = ["fresh", "rotten"]

    for label, cls in enumerate(classes):
        folder = os.path.join(data_dir, split, cls)

        for file in os.listdir(folder):
            if file.lower().endswith(('.jpg', '.png', '.jpeg')):
                X.append(os.path.join(folder, file))
                y.append(label)

    return np.array(X), np.array(y)

DATA_DIR = "D:/Do_An/Data_Final"

X_paths, y = load_dataset(DATA_DIR, "train")

print("Total:", len(X_paths))
print("Fresh:", np.sum(y==0))
print("Rotten:", np.sum(y==1))

Total: 1874
Fresh: 920
Rotten: 954


In [9]:
def load_images(paths):
    X = []
    for p in paths:
        img = load_img(p, target_size=(224,224))
        img = img_to_array(img)
        X.append(img)
    return np.array(X, dtype="float32")

X = load_images(X_paths)
X = preprocess_input(X)

print("Shape:", X.shape)

MemoryError: Unable to allocate 1.05 GiB for an array with shape (1874, 224, 224, 3) and data type float32

In [ ]:
base_model = EfficientNetB0(
    weights="imagenet",
    include_top=False,
    pooling="avg"
)

features = base_model.predict(X, batch_size=32, verbose=1)

print("Feature shape:", features.shape)

: 

In [ ]:
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC())
])

Fitting 3 folds for each of 5 candidates, totalling 15 fits
[CV] END model__dense_units=128, model__dropout_rate=0.5, model__learning_rate=0.0005; total time= 1.1min
[CV] END model__dense_units=128, model__dropout_rate=0.5, model__learning_rate=0.0005; total time= 1.2min
[CV] END model__dense_units=128, model__dropout_rate=0.5, model__learning_rate=0.0005; total time= 1.1min
[CV] END model__dense_units=256, model__dropout_rate=0.5, model__learning_rate=0.0001; total time= 1.1min
[CV] END model__dense_units=256, model__dropout_rate=0.5, model__learning_rate=0.0001; total time= 1.1min
[CV] END model__dense_units=256, model__dropout_rate=0.5, model__learning_rate=0.0001; total time= 1.1min
[CV] END model__dense_units=64, model__dropout_rate=0.3, model__learning_rate=0.001; total time= 1.1min
[CV] END model__dense_units=64, model__dropout_rate=0.3, model__learning_rate=0.001; total time= 1.1min
[CV] END model__dense_units=64, model__dropout_rate=0.3, model__learning_rate=0.001; total time=

In [ ]:
param_grid = {
    "svm__C": [0.1, 1, 10],
    "svm__kernel": ["rbf"],
    "svm__gamma": ["scale", "auto"]
}

grid = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=5,                # 🔥 K-Fold
    scoring="accuracy",
    verbose=2,
    n_jobs=-1
)

grid.fit(features, y)

In [ ]:
print("Best Params:", grid.best_params_)
print("Best Accuracy:", grid.best_score_)

In [ ]:
results = grid.cv_results_

for mean, params in zip(results["mean_test_score"], results["params"]):
    print(f"{mean:.4f} | {params}")


--- FOLD 1/5 ---
Epoch 1/10
185/200 ━━━━━━━━━━━━━━━━━━━━ 10s 732ms/step - accuracy: 0.8390 - loss: 0.3510

ResourceExhaustedError: Graph execution error:

Detected at node PyFunc defined at (most recent call last):
<stack traces unavailable>
MemoryError: Unable to allocate 18.4 MiB for an array with shape (32, 224, 224, 3) and data type float32
Traceback (most recent call last):

  File "d:\App\python\Lib\site-packages\tensorflow\python\ops\script_ops.py", line 269, in __call__
    ret = func(*args)

  File "d:\App\python\Lib\site-packages\tensorflow\python\autograph\impl\api.py", line 643, in wrapper
    return func(*args, **kwargs)

  File "d:\App\python\Lib\site-packages\tensorflow\python\data\ops\from_generator_op.py", line 198, in generator_py_func
    values = next(generator_state.get_iterator(iterator_id))

  File "d:\App\python\Lib\site-packages\keras\src\trainers\data_adapters\generator_data_adapter.py", line 54, in get_tf_iterator
    for batch in self.generator():
                 ~~~~~~~~~~~~~~^^

  File "C:\Users\ADMIN\AppData\Local\Temp\ipykernel_64968\2163453275.py", line 38,

In [ ]:
scores = results["mean_test_score"]

plt.figure()
plt.plot(scores, marker='o')
plt.title("GridSearchCV - EfficientNetB0")
plt.ylabel("Accuracy")
plt.xlabel("Config index")
plt.show()